In [1]:
import os
import tarfile
import polars as pl
import numpy as np
from augmentation.utils.common import read_tsv_from_archive
from augmentation.index import ExhaustiveIndex
from tabulate import tabulate

/home/fedor/Fast_Data_Discovery/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-16 13:42:06,880	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [5]:
a = np.array([0.001, 0.2, 0.3, 0.5])
np.max(np.where(a >= 0.01))

np.int64(3)

In [5]:
checkpoint = 'tsv/c5sh-m8tb.tsv.gz'
# checkpoint = None
start_processing = checkpoint is None
j = 0
target_j = 0
target_counter = 1
files = os.listdir('/mnt/raid0/data/nyc')
counter = 0
for file in files:
    with tarfile.open(f'/mnt/raid0/data/nyc/{file}', 'r:gz') as tar:
        for i, member in enumerate(tar.getmembers()[:None]):
            if member.name.endswith('.tsv.gz'):
                if j != target_j:
                    counter = 0
                    continue
                if checkpoint and not start_processing:
                    if member.name == checkpoint:
                        start_processing = True
                        counter = 0
                        print(f'Starting from checkpoint: {checkpoint}')
                    else:
                        continue

                if (start_processing and counter != target_counter) or (start_processing and j == target_j):
                    print('Found file:', member.name)
                    gz_file = tar.extractfile(member).read()
                    table = read_tsv_from_archive(gz_file)
                    print(table.collect().head())
                    counter += 1
                j += 1

                if counter == target_counter:
                    break

Starting from checkpoint: tsv/c5sh-m8tb.tsv.gz
Found file: tsv/c5sh-m8tb.tsv.gz
shape: (5, 15)
┌───────────┬───────────┬──────┬───────────┬───┬──────────────────┬─────────────┬───────┬──────────┐
│ level_4_2 ┆ level_1_1 ┆ year ┆ level_1_2 ┆ … ┆ mean_scale_score ┆ level_3_4_1 ┆ grade ┆ category │
│ ---       ┆ ---       ┆ ---  ┆ ---       ┆   ┆ ---              ┆ ---         ┆ ---   ┆ ---      │
│ f64       ┆ i64       ┆ i64  ┆ f64       ┆   ┆ i64              ┆ i64         ┆ str   ┆ str      │
╞═══════════╪═══════════╪══════╪═══════════╪═══╪══════════════════╪═════════════╪═══════╪══════════╡
│ 25.0      ┆ 2908      ┆ 2006 ┆ 8.2       ┆ … ┆ 675              ┆ 27134       ┆ 3     ┆ Female   │
│ 21.1      ┆ 3283      ┆ 2006 ┆ 9.3       ┆ … ┆ 670              ┆ 24935       ┆ 4     ┆ Female   │
│ 16.7      ┆ 4284      ┆ 2006 ┆ 11.7      ┆ … ┆ 661              ┆ 22686       ┆ 5     ┆ Female   │
│ 11.7      ┆ 5893      ┆ 2006 ┆ 16.4      ┆ … ┆ 651              ┆ 19273       ┆ 6     ┆ Female 

In [6]:
df = table.collect()
df.head(), df.shape

(shape: (5, 15)
 ┌───────────┬───────────┬──────┬───────────┬───┬──────────────────┬─────────────┬───────┬──────────┐
 │ level_4_2 ┆ level_1_1 ┆ year ┆ level_1_2 ┆ … ┆ mean_scale_score ┆ level_3_4_1 ┆ grade ┆ category │
 │ ---       ┆ ---       ┆ ---  ┆ ---       ┆   ┆ ---              ┆ ---         ┆ ---   ┆ ---      │
 │ f64       ┆ i64       ┆ i64  ┆ f64       ┆   ┆ i64              ┆ i64         ┆ str   ┆ str      │
 ╞═══════════╪═══════════╪══════╪═══════════╪═══╪══════════════════╪═════════════╪═══════╪══════════╡
 │ 25.0      ┆ 2908      ┆ 2006 ┆ 8.2       ┆ … ┆ 675              ┆ 27134       ┆ 3     ┆ Female   │
 │ 21.1      ┆ 3283      ┆ 2006 ┆ 9.3       ┆ … ┆ 670              ┆ 24935       ┆ 4     ┆ Female   │
 │ 16.7      ┆ 4284      ┆ 2006 ┆ 11.7      ┆ … ┆ 661              ┆ 22686       ┆ 5     ┆ Female   │
 │ 11.7      ┆ 5893      ┆ 2006 ┆ 16.4      ┆ … ┆ 651              ┆ 19273       ┆ 6     ┆ Female   │
 │ 8.0       ┆ 6260      ┆ 2006 ┆ 17.0      ┆ … ┆ 643             

In [4]:
worker = ExhaustiveIndex(feature_selection_table_name='', overlap_table_name='')

In [ ]:
result = worker.index_table(table, 0)

In [1]:
import requests
from augmentation.retrieval import AurumJoinDiscovery

worker = AurumJoinDiscovery(api_host='localhost', api_port=5000)
source_name = 'inspections_regression.csv'
payload = {
    "table_path": source_name,
    "lake": "nyc"
}
request = requests.post(f'http://localhost:5000/similar_tables', json=payload)
response = request.json()

dict_keys(['similar_tables'])

In [3]:
from experiments.base_tables.base_table_preprocessing import PreProcessor

preprocessor = PreProcessor(
    'experiments/base_tables/inspections_regression/inspections_regression.csv',
    'experiments/base_tables/inspections_regression/splits.json',
    split_index=0
)
X, query_col, target, nan_mask = preprocessor.run(imputation='simple', encode=False, augmented=False)
X.write_csv('experiments/base_tables/inspections_regression/inspections_regression_preprocessed.csv')

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
from augmentation.feature_selection.algorithms.metam import MetamAugmenter

augmenter = MetamAugmenter()
augmenter.run(
    task='regression',
    path='/mnt/raid0/data/extracted/nyc',
    query_data='inspections_regression_preprocessed.csv',
    filepath='experiments/base_tables/inspections_regression/join_paths/metam.csv',
    class_attr='SCORE',
    epsilon=0.9,
    theta=0.95,
    uninfo=0,
    orig_metric=0.5,
    output_path='experiments/base_tables/inspections_regression/inspections_regression_metam.csv',
    sep_lake='\t',
    sep_query=','
)

In [3]:
from augmentation.feature_selection.algorithms.arda import ArdaAugmenter

arda = ArdaAugmenter()
left_table, fetch_time, augmentation_time = arda.run(
    join_paths_df_path='experiments/base_tables/inspections_regression/join_paths/arda.csv',
    query_table=X,
    query_column_name=query_col,
    data_lake_folder='/mnt/raid0/data/extracted/nyc',
    base_node_id='inspections_regression_preprocessed.csv',
    target_column_name='SCORE',
    sample_size=3000,
    regression=True,
    sep_lake='\t'
)

100%|██████████| 19/19 [00:33<00:00,  1.74s/it]


In [ ]:
left_table

In [ ]:
from augmentation.feature_selection.algorithms.autofeat import AutoFeat

autofeat = AutoFeat()

aug_df = autofeat.run(
    join_paths_df_path='experiments/base_tables/inspections_regression/join_paths/arda.csv',
    query_column_name='BORO',
    data_lake_folder='/mnt/raid0/data/extracted/nyc',
    problem_type='regression',
    base_table_label='inspections_regression',
    base_table_id='inspections_regression_preprocessed.csv',
    target_column_name='SCORE',
    base_table_sep=',',
    lake_table_sep='\t'
)

In [4]:
best_path = 'inspections_regression_preprocessed.csv--inspections_regression_preprocessed.csv-BORO-boro-59kj-x8nc.csv--inspections_regression_preprocessed.csv-BORO-borough-ebb7-mvp5.csv'
paths = best_path.split('--')[1:]
from_table = 'inspections_regression_preprocessed.csv'
for path in paths:
    print(path.split('-'))

['inspections_regression_preprocessed.csv', 'BORO', 'boro', '59kj', 'x8nc.csv']
['inspections_regression_preprocessed.csv', 'BORO', 'borough', 'ebb7', 'mvp5.csv']


In [ ]:
from autogluon.tabular import TabularPredictor

TabularPredictor().model_names()

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from augmentation.feature_selection.algorithms.kitana import KitanaAugmenter
from experiments.base_tables.base_table_preprocessing import PreProcessor

preprocessor = PreProcessor(
    'experiments/base_tables/inspections_regression/inspections_regression.csv',
    'experiments/base_tables/inspections_regression/splits.json',
    split_index=0
)
X, query_col, target, nan_mask = preprocessor.run(imputation='simple', encode=False, augmented=False)
X.write_csv('experiments/base_tables/inspections_regression/inspections_regression_preprocessed.csv')

kitana = KitanaAugmenter()
kitana.run(
    join_paths_df_path='experiments/base_tables/inspections_regression/join_paths/arda.csv',
    query_column_name=query_col,
    query_table_path='experiments/base_tables/inspections_regression/inspections_regression_preprocessed.csv',
    lake_data_path='/mnt/raid0/data/extracted/nyc',
    target_column_name='SCORE',
    buyer_sep=',',
    seller_sep='\t',
    n_iter=10
)

In [ ]:
import pandas as pd
from augmentation.feature_selection.algorithms.cocoa import CocoaAugmenter

dataset = pd.read_csv('experiments/base_tables/inspections_regression/inspections_regression.csv')
cocoa = CocoaAugmenter()
cocoa.run(
    distinct_tokens_table='nyc_cocoa_distinct_tokens',
    main_tokenized_table='nyc_cocoa_main_tokenized',
    max_column_table='nyc_cocoa_max_column',
    order_index_table='nyc_cocoa_ordered_index',
    query_table_path='experiments/base_tables/inspections_regression/inspections_regression_preprocessed.csv',
    query_column='STREET',
    target_column='SCORE',
    top_k_joinable=5,
    top_k_correlated=5
)

In [ ]:
# from augmentation.feature_selection.algorithms.qcr import QcrAugmenter

# qcr = QcrAugmenter()
# qcr.run(
#     50,
#     'BORO',
#     'SCORE',
#     'nyc_qcr_index',
#     '/mnt/raid0/data/extracted/nyc',
#     'experiments/base_tables/inspections_regression/inspections_regression_preprocessed.csv',
#     ',',
#     '\t'
# )

In [ ]:
# qcr.run(
#     top_k=50,
#     query_column_name='BORO',
#     target_column_name='SCORE',
#     index_table_name='nyc_qcr_index',
#     data_lake_path='/mnt/raid0/data/extracted/nyc',
#     query_table_path='experiments/base_tables/inspections_regression/inspections_regression_preprocessed.csv',
#     lake_table_sep='\t',
#     query_table_sep=','
# )